# 1.0 Load packages and data

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load data
df = pd.read_csv("second_resampled_tus.csv")

In [ ]:
# Load data
df1 = pd.read_csv("cluster_merge.csv")

In [ ]:
df.head()

# 2.0 Household Behaviour

## 2.1 What is the overall pattern of households?

In [ ]:
# Create dataframe aggregated by time_bin
df_behaviour_overall = (df.groupby('time_bin', observed=True)
                     .agg(hh_count=('mainid', 'nunique'),  # use unique mainid to avoid double-counting households
                          total_bin_duration=('bin_duration_minutes', 'sum'))
                       ).reset_index()  ## calculate the sum of duration

In [ ]:
# Compute metrics
# Get number of households in data
n_hhold = df['mainid'].nunique()  

# calculate percentage of households active during this time bin (doing any activity)
df_behaviour_overall['percentage_hh'] = (
    df_behaviour_overall['hh_count'] / n_hhold) * 100

# Average duration per household across all households (including non-participants as zeros)
df_behaviour_overall['avg_duration_per_hh'] = df_behaviour_overall['total_bin_duration'] / n_hhold

In [ ]:
# View data aggregation
df_behaviour_overall.head()

In [ ]:
# Visualise aggregated data
# Plot general behaviour for energy-using activities
plt.figure(figsize=(16, 6))  # set the figure layout
plt.plot(df_behaviour_overall['time_bin'], df_behaviour_overall['percentage_hh'],
         color="#36454F", marker='o', markersize=4)

plt.xlabel('Time of Day', fontsize=12) # label x-axis
plt.ylabel('Household Participation Rate (%)', fontsize=12)   # label y-axis
plt.title('Overall Household Participation Rate Throughout the Day', fontsize=14)  # add title
plt.xticks(rotation=45, ha='right')   # rotate the x-axis ticks for readability
plt.grid(True, alpha=0.3)            # add grid lines

plt.tight_layout()   # improve padding 
plt.savefig("Overall Household Participation Rate Throughout the Day.png") # Saving the plot as a png file
plt.show()

Two peaks of household engagement are  observed in the morning and later in the evenings. Participation rate during the morning hours, 05:30-08:00 am, is low that in the evening; morning peak increased from 5% to about 28% of households, while evening peak from 15:00-21:59 pm, rose from about 12% to 58% of households before declining to 10% at 23:29pm. These suggests that overall behaviour would be activity-type dependent.

## 2.2 How are the activities engaged?

In [ ]:
# Re-aggregate the data to include activity description column
df_activities = (df.groupby(['time_bin', 'starttime_of_day_min', 'activity_description'], observed=True)
                 .agg(hh_count=('mainid', 'nunique'),          # use unique mainid to avoid double-counting households
                      total_bin_duration=('bin_duration_minutes', 'sum')      # add all duration per household within any time period
                     ).reset_index())

In [ ]:
# Compute metrics
# Calculate percentage of all households participating in this activity during a time bin
df_activities['percentage_hh'] = (df_activities['hh_count'] / n_hhold) * 100  # n_hhold is the same as in the overall behaviour above

# Average duration per household across all households (including those not participating at a give time)
df_activities['avg_duration_per_hh'] = df_activities['total_bin_duration'] / n_hhold

# Sort the dataframe
df_activities = df_activities.sort_values(['activity_description', 'time_bin']).reset_index(drop=True)

In [ ]:
df_activities.head()

In [ ]:
# Extract the unique activity names from the dataset
activities = df_activities["activity_description"].unique()

# Loop through each activity to create a separate line chart
for activity in activities:
    # Filter the dataset to include only records for the current activity
    df_plot = df_activities[df_activities["activity_description"] == activity].copy()
    
    # Sort the data by the activity start time to ensure the x-axis is in chronological order
    df_plot = df_plot.sort_values("starttime_of_day_min")
    
    # Create a new figure for the current activity
    plt.figure(figsize=(14, 5))
    
    # Plot household participation rate across time of day for the current activity
    sns.lineplot(data=df_plot, x="time_bin", y="percentage_hh",
                 estimator=None, linewidth=2, color="#36454F", marker="o")
    
    # Add a descriptive title showing the activity being plotted
    plt.title(f"Households' Patterns in Engaging with {activity} Throughout the Day",
              fontsize=14, fontweight='bold')
    
    # Label the x-axis and y-axis
    plt.xlabel("Time of Day", fontsize=12)
    plt.ylabel("Households Participation Rate (%)", fontsize=12)
    
    # Rotate x-axis labels to improve readability
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(True, alpha=0.3)     # Add a light grid to make the chart easier to read
    plt.tight_layout()    # Adjust layout to prevent label overlap
    
    # Save the figure using the activity name in the file name
    plt.savefig(f"Households' Patterns in Engaging with {activity} Throughout the Day.png")
    plt.show()    # Display the chart

Washing & dressing, House Cleaning, and Laundry are predominantly morning activities, while dishwashing, food preparation, and watching TV are evening activities.

## 2.3 What is the behaviours of the household archetypes?

I will create a function that filters the socioeconomic features and returns their time patterns and participation metrics. It aggregates household activity durations within each time bin (and optional groupings), then computes the % of households participating within each group.

In [ ]:
# Create aggregation function
def aggregate_by_feature(df, group_by, include_activity=False):
    
    # Build grouping columns
    group_cols = ['time_bin', 'starttime_of_day_min']
    
    if include_activity:           # if activity is needed, append activity columne to the dataframe
        group_cols.append('activity_description')
    
    group_cols.extend(group_by)  # make the function flexible should you need any features to be filtered
    
    # Aggregate by groups
    df_result = df.groupby(group_cols, observed=True).agg(
        hh_count=('mainid', 'nunique'),           # ensure the households are unique
        total_duration=('bin_duration_minutes', 'sum')            # get the sum of durations of activities withing each time bin
    ).reset_index()

    # Compute metrics
    # Calculate percentage using first grouping column
    # get total unique households for each value of group_by[0]
    total_per_group = (
        df.groupby(group_by[0], observed=True)['mainid']
          .nunique())
    
    # Map each row’s group_by[0] value to its total household count (denominator)
    # and compute the percentage of households participating in that time_bin
    df_result['percentage_hh'] = (df_result['hh_count'] / df_result[group_by[0]]
        .map(total_per_group)) * 100
    
    # Sort results for readability and plotting
    df_result = df_result.sort_values(group_cols).reset_index(drop=True)
    
    return df_result           # Return the aggregated table

In [ ]:
# Aggregate the data for the clusters
df_clusters = aggregate_by_feature(df, ['cluster'])
df_clusters.head(3)

In [ ]:
# Set the colors to ensure contrast
n_colors = df_clusters['cluster'].nunique()
palette = sns.color_palette('tab20', n_colors)

In [ ]:
# Sort the cluster-level data by time of day so the x-axis follows chronological order
df_plot = df_clusters.sort_values("starttime_of_day_min")
    
# Create a figure for the cluster comparison plot
plt.figure(figsize=(18, 8))
    
# Plot household participation rate across time of day,
# with a separate line for each cluster
sns.lineplot(data=df_plot, x="time_bin", y="percentage_hh",
             hue="cluster", palette=palette, estimator="mean", linewidth=2)
    
# Add a descriptive chart title plus label the x-axis and y-axis
plt.title("Energy Use Patterns of Household Clusters")
plt.xlabel("Time of Day", fontsize=12)  
plt.ylabel("Households Participation Rate (%)", fontsize=12)

# Rotate x-axis labels to improve readability
plt.xticks(rotation=45, ha='right', fontsize=10)

# Add a legend below the chart and arrange items across columns
plt.legend(title="Cluster", loc="upper center",
    bbox_to_anchor=(0.5, -0.3),  # centre the legend below the plot
    ncol=3,                      # arrange legend entries across 3 columns
    frameon=True)

plt.subplots_adjust(bottom=0.35)  # Increase the bottom margin to make space for the legend
plt.grid(True, alpha=0.3)# Add a light grid to improve readability
plt.tight_layout()  # Adjust layout to prevent overlap between plot elements

# Save the chart
plt.savefig("Energy Use Patterns of Household Clusters.png", 
            dpi=300, bbox_inches="tight")

# Display the plot
plt.show()

## 2.4 Overall Cluster Behaviour by Activity

In [ ]:
# Aggregate the data for the clusters
df_cluster_activity = aggregate_by_feature(df, ['cluster'],
                                           include_activity=True)
df_cluster_activity.head(3)

In [ ]:
# Loop through each activity to create a separate plot
for activity in activities:
    # Filter the dataset to include only records for the current activity
    df_plot = df_cluster_activity[
        df_cluster_activity["activity_description"] == activity].copy()

    # Sort the data by start time so the x-axis appears in chronological order
    df_plot = df_plot.sort_values("starttime_of_day_min")

    # Count the number of unique clusters to determine how many colours are needed
    n_colors = df_plot["cluster"].nunique()

    # Generate a colour palette for the clusters
    palette = sns.color_palette("tab20", n_colors)
    
    # Create a figure for the current activity
    plt.figure(figsize=(18, 8))

    # Plot household participation rate over time of day,
    # using a separate line for each cluster
    sns.lineplot(data=df_plot, x="time_bin", y="percentage_hh",
                 hue="cluster", estimator="mean", linewidth=2, palette=palette)
    
    # Add a descriptive title showing the activity and label the x-axis and y-axis
    plt.title(f"Households' Patterns in Engaging with {activity} Throughout the Day",
              fontsize=14, fontweight="bold")
    plt.xlabel("Time of Day", fontsize=12)
    plt.ylabel("Households Participation Rate (%)", fontsize=12)

    # Rotate x-axis labels to improve readability
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.yticks(fontsize=10)

    # Add a light grid
    plt.grid(True, alpha=0.3)

    # Add a legend below the chart
    plt.legend(title="Cluster", loc="upper center",
               bbox_to_anchor=(0.5, -0.3), ncol=3, frameon=True)

    plt.subplots_adjust(bottom=0.35)    # Increase the bottom margin to make space for the legend
    plt.tight_layout()    # Adjust the layout so labels and legend do not overlap

    # Save the figure as a high-resolution image
    plt.savefig(f"Households' Patterns in Engaging with {activity} Throughout the Day.jpg",
                dpi=300, bbox_inches="tight")

    plt.show()    # Display the plot
    plt.close()    # Close the figure to free memory before the next loop iteration

In [ ]:
activity_colors = {'Dishwashing':        "#1f77b4",  # blue
                   'Food Preparation':   "#d62728",  # red
                   'Heating House':      "#ffdd57",  # yellow
                   'House Cleaning':     "#2ca02c",  # green
                   'Laundry':            "#ff7f0e",  # orange
                   'Washing & Dressing': "#9467bd",  # purple
                   'Watching TV':        "#17becf"   # teal
}

# Extract the unique cluster labels from the dataset
clusters = df_cluster_activity["cluster"].unique()

In [ ]:
# Loop through each cluster to create a separate stacked area chart
for cluster in clusters:
    # Filter the dataset to include only records for the current cluster
    df_plot = df_cluster_activity[df_cluster_activity["cluster"] == cluster].copy()

    # Sort the data by start time so the x-axis follows chronological order
    df_plot = df_plot.sort_values("starttime_of_day_min")
    
    # Reshape the data so each row represents a time bin and each column represents an activity
    df_pivot = df_plot.pivot_table(index="time_bin", columns="activity_description",
                                   values="percentage_hh", aggfunc="mean")

    # Convert each row to percentages so that activity shares sum to 100% within each time bin
    df_pivot = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100
    df_pivot = df_pivot.fillna(0)     # Replace missing values with 0 after normalisation
    
    # Retrieve the predefined colour for each activity in the same order as the pivot table columns
    palette = [activity_colors[activity] for activity in df_pivot.columns]
    
    # Create the figure and axis for the stacked area chart
    fig, ax = plt.subplots(figsize=(14, 6))

    # Plot the activity distribution over time as a stacked area chart
    ax.stackplot(range(len(df_pivot)), df_pivot.T.values,
                 labels=df_pivot.columns, colors=palette, alpha=0.85)
    
    ax.set_xlim(0, len(df_pivot) - 1)     # Set the x-axis range to cover all time bins
    ax.set_xticks(range(len(df_pivot)))    # Show one tick per time bin
    ax.set_xticklabels(df_pivot.index, rotation=45, ha="right", fontsize=9)    # Label the x-axis ticks using the time bin values
    ax.set_ylim(0, 100)    # Set the y-axis range from 0 to 100 percent

    # Add a descriptive title for the current cluster and label the x-axis and y-axis
    ax.set_title(f"Activity Engagement Distribution for {cluster}",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Time of Day", fontsize=12)
    ax.set_ylabel("Share of Activity (%)", fontsize=12)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3),     # Add a legend below the chart
              ncol=3, frameon=True, title="Activity")
    plt.subplots_adjust(bottom=0.35)    # Increase the bottom margin to make space for the legend
    plt.tight_layout()    # Adjust layout to prevent overlap between labels and legend

    # Save the figure using a cleaned file name
    plt.savefig(f"Activity_Distribution_{cluster}.png".replace("/", "_"),
                dpi=300, bbox_inches="tight")
    plt.show()    # Display the chart
    plt.close()    # Close the figure to free memory before the next iteration

I will represent it on a heatmap

In [ ]:
# Loop through each cluster to create a separate heatmap
for cluster in clusters:
    # Filter the dataset to include only records for the current cluster
    df_plot = df_cluster_activity[df_cluster_activity["cluster"] == cluster].copy()

    # Sort the data by start time so the time order is consistent
    df_plot = df_plot.sort_values("starttime_of_day_min")
    
    # Reshape the data so activities form the rows
    # and time bins form the columns
    df_pivot = df_plot.pivot_table(index="activity_description", columns="time_bin",
                                   values="percentage_hh", aggfunc="mean")
    df_pivot = df_pivot.fillna(0)    # Replace missing values with 0 so all activity-time combinations can be displayed
    
    # Create a figure, adjusting the height based on the number of activities
    fig, ax = plt.subplots(figsize=(18, max(4, len(df_pivot) * 0.8)))
    
    # Plot the heatmap showing participation rate by activity and time of day
    sns.heatmap(df_pivot, cmap="YlOrRd", linewidths=0.5, linecolor="white",
                cbar_kws={"label": "Participation Rate (%)"}, ax=ax)
    
    # Add a descriptive title for the current cluster and label the x-axis and y-axis
    ax.set_title(f"Activity Engagement Heatmap for {cluster}",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Time of Day", fontsize=12)
    ax.set_ylabel("Activity", fontsize=12)

    # Rotate x-axis labels for readability and keep y-axis labels horizontal
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
    
    plt.tight_layout()    # Adjust the layout to prevent overlap

    # Save the figure using a cleaned file name
    plt.savefig(f"Activity_Heatmap_{cluster}.png".replace("/", "_"), 
                dpi=300, bbox_inches="tight")

    plt.show()     # Display the heatmap
    plt.close()    # Close the figure to free memory before the next iteration

# 3.0 Forced Typology Analysis

The forced typology is the approach where domain knowledge is leveraged to identify common households types in the society, specifically the income, children, unemployment, and retirement presence. Although the bias in this is that it is deterministic, it can also point to which of these factors  makes a household rigid.

In [ ]:
# Define children presence
df1['is_young'] = (
    (df1['under_age5'] > 0) |
    (df1['age5to11'] > 0) |
    (df1['age12to16'] > 0))

In [ ]:
# Define typologies according to income label and categories, employment grouping and children

typology_rules = [
    # High income
    ('High income', ['high income', 'very high income'], 'fully employed', True), # High income is true, and fully employed is true
    ('High income', ['high income', 'very high income'], 'fully employed', False), # High income is true, but fully employed is false
    ('High income', ['high income', 'very high income'], 'partly employed', True),
    ('High income', ['high income', 'very high income'], 'partly employed', False),# High income is true, but partly employed is true
    ('High income', ['high income', 'very high income'], 'fully retired', True),
    ('High income', ['high income', 'very high income'], 'fully retired', False), # High income is true, but fully retired is true
    ('High income', ['high income', 'very high income'], 'partly retired', True),
    ('High income', ['high income', 'very high income'], 'partly retired', False), # High income is true, but partly retired is true
    ('High income', ['high income', 'very high income'], 'fully unemployed', True),
    ('High income', ['high income', 'very high income'], 'fully unemployed', False), # High income is true, but fully employed is true
    ('High income', ['high income', 'very high income'], 'partly unemployed', True),
    ('High income', ['high income', 'very high income'], 'partly unemployed', False), # High income is true, but partly employed is true
    
    # Med income
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'fully employed', True),  # Med income is true and fully employed is true
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'fully employed', False),
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'partly employed', True), # Med income is true and fully employed is true
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'partly employed', False),
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'fully retired', True), # Med income is true and fully employed is true
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'fully retired', False),
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'partly retired', True), # Med income is true and partly employed is true
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'partly retired', False),
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'fully unemployed', True), # Med income is true and fully employed is true
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'fully unemployed', False),
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'partly unemployed', True), # Med income is true and partly unemployed is true
    ('Med income', ['lower middle income','middle income', 'upper middle income'], 'partly unemployed', False),
    
    # Low income
    ('Low income', ['low income'], 'fully employed', True),
    ('Low income', ['low income'], 'fully employed', False),
    ('Low income', ['low income'], 'partly employed', True),
    ('Low income', ['low income'], 'partly employed', False),
    ('Low income', ['low income'], 'fully retired', True),
    ('Low income', ['low income'], 'fully retired', False),
    ('Low income', ['low income'], 'partly retired', True),
    ('Low income', ['low income'], 'partly retired', False),
    ('Low income', ['low income'], 'fully unemployed', True),
    ('Low income', ['low income'], 'fully unemployed', False),
    ('Low income', ['low income'], 'partly unemployed', True),
    ('Low income', ['low income'], 'partly unemployed', False),
]

## 3.1 Time periods without activities

In [ ]:
# Calculate metrics for each typology (in order)
# Create empty container
results = []

for idx, (income_label, income_cats, employment, has_children) in enumerate(typology_rules, 1):
    
    # Filter data
    subset = df1[
        (df1['income_category'].isin(income_cats)) &
        (df1['household_employment_group'] == employment) &
        (df1['is_young'] == has_children)]
    
    # Calculate metrics
    n = len(subset)
    # Calculate the percentage per sample
    metric_percent = (len(subset) / len(df1)) * 100
    
    # Create readable typology name
    children = "Has young children" if has_children else "No young children"
    typology_name = f"{income_label} AND {employment} AND {children}"

    # Append the metrics and replace zero with nan
    results.append({
        'Order': idx,
        'Typology': typology_name,
        'N_households': n,
        'Pct_of_total': round(metric_percent, 2),
        'Mean_usage_std': round(subset['usage_std'].mean(), 3) if n > 0 else np.nan,
        'Mean_load_factor': round(subset['load_factor'].mean(), 3) if n > 0 else np.nan,
        'Std_load_factor': round(subset['load_factor'].std(), 3) if n > 0 else np.nan,
        'Mean_daytime_lf': round(subset['daytime_lf'].mean(), 3) if n > 0 else np.nan,
        'Std_daytime_lf': round(subset['daytime_lf'].std(), 3) if n > 0 else np.nan,
        'Mean_evening_lf': round(subset['evening_lf'].mean(), 3) if n > 0 else np.nan,
        'Std_evening_lf': round(subset['evening_lf'].std(), 3) if n > 0 else np.nan,
        'Mean_offpeak_lf': round(subset['offpeak_lf'].mean(), 3) if n > 0 else np.nan,
        'Std_offpeak_lf': round(subset['offpeak_lf'].std(), 3) if n > 0 else np.nan,
    })

In [ ]:
# Create DataFrame
df_typologies = pd.DataFrame(results)
df_typologies

In [ ]:
# Save to CSV
df_typologies.to_csv('forced_typology_metrics_ordered.csv', index=False)

In [ ]:
# Get typologies from the dataframe
typologies = df_typologies["Typology"].unique()

In [ ]:
# Loop through each typology to create a separate bar chart
for typology in typologies:
    # Filter the dataset to include only rows for the current typology
    df_plot = df_typologies[df_typologies["Typology"] == typology]

    # Get the number of households for the current typology
    n_hh = df_plot["N_households"].iloc[0]
    
    # Select the mean load factor values for the four time periods and transpose them so each period becomes a separate bar
    df_means = df_plot[
        ["Mean_load_factor", "Mean_daytime_lf", "Mean_evening_lf", "Mean_offpeak_lf"]].T

    # Replace the original row labels with clearer time period names
    df_means.index = ["Whole Day", "Daytime", "Evening", "Offpeak"]

    # Rename the single column to make the plotted value clear
    df_means.columns = ["Load Factor"]
    
    # Create a bar chart of load factor means across time periods
    ax = df_means.plot(kind="bar", figsize=(13, 4), width=0.5, legend=False,
                   color=["#1f77b4", "#ff7f0e", "#d62728", "#2ca02c"])

    # Add a title showing the typology name and household count
    ax.set_title(f"{typology}  (N = {n_hh} households)", fontsize=11)

    # Label the x-axis and y-axis
    ax.set_xlabel("Time Period")
    ax.set_ylabel("Load Factor Mean")

    # Keep x-axis labels horizontal for readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")

    plt.tight_layout()    # Adjust layout to prevent overlap

    # Save the figure using the typology name in the file name
    plt.savefig(f'lf_{typology.replace(" ", "_")}.png',
                dpi=150, bbox_inches="tight")
    plt.show()    # Display the chart

High======

High income AND fully employed AND Has children are less rigid across teh whole day, daytime and offpeak period. They are very rigid in the evening period. While High income AND fully employed and No young children are rigid across the whole day, in the daytime and offpeak periods they are less rigid, but very rigid in the evening periods.

High income AND partly employed AND Has young children are less rigid across the whole day, daytime, and offpeak period, while very rigid in the evening period. High income AND partly employed AND No young children is moderately rigid across the whole day, but less rigid in the daytime and offpeak periods, while very rigid in the evening.

High income AND fully retired AND No young children and High income AND partly retired AND Has young children are rigid across the day, including daytime and evening, while offpeak was less rigid. High income AND partly retired AND No young children is rigid across the day and the daytime, less rigid in the offpeak, and very rigid in the evening peak.

High income AND partly unemployed AND Has young children are less rigid across the day; the daytime have almost similar measure as the across the day. However, the evening is very rigid. High income AND partly unemployed AND No young children are less rigid across the day, and particularly in daytime and offpeak, but very rigid in the evening.

Medium=======

Med income AND fully employed AND Has young children are moderately rigid across the whole day. They are less rigid in the offpeak, rigid in the daytime, and very rigid in the evening. Med income AND fully employed AND No young children are rigid across the day, and particularly in the daytime, while being very rigid in the evening and less rigid in the offpeak.

Med income AND partly employed AND Has young children are less rigid across the whole day, the daytime and offpeak, but very rigid in the evening. Med income AND partly employed AND No young children are moderate across the day; their daytime period, is moderate, the offpeak is less rigid, while, the evening is very rigid.

Med income AND fully retired AND Has young children are very rigid across the day; the evenings are highly very rigid, while mornings are less rigid. Med income AND fully retired AND No young children are very rigid across the day; the evenings are higly very rigid, mornings are very rigid, whild the offpeaks are rigid.

Med income AND partly retired AND Has young children are very rigid across the day; the evenings are higly very rigid, mornings are very rigid, whild the offpeaks are less rigid. Med income AND partly retired AND No young children is very rigid across the day; the daytime is rigid, evening is very rigid, while offpeak are moderate.

Med income AND partly unemployed AND Has young children is rigid across the day; while daytime and evening are very rigid, the offpeak is less rigid. Med income AND partly unemployed AND No young children is rigid across the day; while daytime and evening are very rigid, the offpeak is moderately rigid.

Low======

Low income AND fully employed AND Has young children is moderately rigid across the day, the daytime is rigid, evening very rigid, but the offpeak is less rigid. Low income AND fully employed AND No young children is moderately rigid across teh day; less rigid in offpeak, moderate in the daytime, and very rigid in the evening.

Low income AND partly employed AND Has young children is moderately rigid across the day; daytimes are moderately rigid, offpeak are less rigid, and evenings are very rigid. Low income AND partly employed AND No young children is rigid across the day; the daytime is rigid, less rigid in the offpeak, and very rigid in the evening.

Low income AND fully retired AND No young children is very rigid across the day; while very rigid in the daytime, evenings are highly very rigid, and offpeak are less rigid. 

Low income AND partly retired AND Has young children is rigid across the day; daytime and evenings are very rigid. Low income AND partly retired AND No young children is very rigid across the day; daytime is very rigid, offpeak is less rigid, and evenings are highly very rigid.

Low income AND fully unemployed AND Has young children are very less rigid across the day; they are just less rigid in the evening. Low income AND fully unemployed AND No young children is rigid across the day; the offpeaks are very less rigid, the daytime rigid, but the evening very rigid.

Low income AND partly unemployed AND Has young children is moderate across the day; the daytime is moderately rigid, offpeak is less rigid, and very rigid in the evening. Low income AND partly unemployed AND No young children is rigid across the day; while daytime and eveinings are very rigid, the offpeak is rigid.

## 3.2 Time periods and activities

I will create a function that calculates activity load factors (mean and std) for each specific typology across time periods.

In [ ]:
# Label each household with its typology
def assign_typology(row):
    # Loop through each predefined typology and assign an index starting from 1
    for idx, (income_label, income_cats, employment, has_children) in enumerate(typology_rules, 1):
        # Check whether the row matches the current typology definition:
        if (row["income_category"] in income_cats and
            row["household_employment_group"] == employment and
            row["is_young"] == has_children ):
            # Create a readable label describing child status
            children_str = "Has young children" if has_children else "No young children"

            # Return both the typology index and a descriptive typology label
            return idx, f"{income_label} AND {employment} AND {children_str}"

    # Return None values if the row does not match any typology
    return None, None

In [ ]:
# Create order
df1[['typo_order', 'typo_name']] = df1.apply(
    assign_typology, axis=1, result_type='expand')

In [ ]:
# Define peak times
# Infer bin size in minutes
time_bins_min = np.arange(0, 1440, 30, dtype=int) 

# morning peak run between 07:00 am and 11:00 am
day_engagement = time_bins_min[
    (time_bins_min >= 7*60) & 
    (time_bins_min < 16*60)]

# Evening peaks run between 4:00pm and 23:00pm
evening_engagement = time_bins_min[
    (time_bins_min >= 16*60) &
    (time_bins_min < 23*60)]

# Off peak run between 11am and 4pm
offpeak_engagement = time_bins_min[
    ((time_bins_min >= 23*60) &
    (time_bins_min < 24*60)) |
    ((time_bins_min >= 0*60) &
    (time_bins_min < 7*60))]

In [ ]:
# Calculate activity load factors for a given typology
def calculate_activity_lf(df_expanded, mainids, day_engagement, evening_engagment, offpeak_engagement):
    
    # Filter the expanded dataset to include only households belonging to the typology
    df_typology = df_expanded[df_expanded["mainid"].isin(mainids)]
    
    # Overall day -----------------------------------------------------------------
    # Group by household and activity, then calculate the maximum and mean duration
    all_day = df_typology.groupby(["mainid", "activity_description"])["bin_duration_minutes"].agg(
        [("max_bin", "max"), ("mean_bin", "mean")]).reset_index()

    # Calculate the load factor for each household-activity combination
    # as mean duration divided by maximum duration
    all_day["all_day_lf"] = all_day["mean_bin"] / all_day["max_bin"].replace(0, np.nan)

    # Aggregate to activity level by taking the mean and standard deviation
    # of the household load factors
    all_day_lf = all_day.groupby("activity_description")["all_day_lf"].agg(
        [("day_lf_mean", "mean"), ("day_lf_std", "std")]).reset_index()
    
    # Morning peak -----------------------------------------------------------------
    # Filter records to the selected daytime engagement period
    daytime_data = df_typology[df_typology["starttime_of_day_min"].isin(day_engagement)]

    # Group by household and activity, then calculate the maximum and mean duration
    daytime_hh = daytime_data.groupby(["mainid", "activity_description"])["bin_duration_minutes"].agg(
        [("max_bin", "max"), ("mean_bin", "mean")]).reset_index()
    
    # Calculate the load factor for each household-activity combination
    daytime_hh["lf"] = daytime_hh["mean_bin"] / daytime_hh["max_bin"].replace(0, np.nan)
    
    # Aggregate to activity level
    daytime_lf = daytime_hh.groupby("activity_description")["lf"].agg(
        [("daytime_lf_mean", "mean"), ("daytime_lf_std", "std")]).reset_index()
    
    # Evening peak -----------------------------------------------------------------
    # Filter records to the selected evening engagement period
    evening_data = df_typology[df_typology["starttime_of_day_min"].isin(evening_engagement)]

    # Group by household and activity, then calculate the maximum and mean duration
    evening_hh = evening_data.groupby(["mainid", "activity_description"])["bin_duration_minutes"].agg(
        [("max_bin", "max"), ("mean_bin", "mean")]).reset_index()
    
    # Calculate the load factor for each household-activity combination
    evening_hh["lf"] = evening_hh["mean_bin"] / evening_hh["max_bin"].replace(0, np.nan)
    
    # Aggregate to activity level
    evening_lf = evening_hh.groupby("activity_description")["lf"].agg(
        [("evening_lf_mean", "mean"), ("evening_lf_std", "std")]).reset_index()
    
    # Off-peak -----------------------------------------------------------------
    # Filter records to the selected off-peak engagement period
    off_peak_data = df_typology[
        df_typology["starttime_of_day_min"].isin(offpeak_engagement)]

    # Group by household and activity, then calculate the maximum and mean duration
    offpeak_hh = off_peak_data.groupby(["mainid", "activity_description"])["bin_duration_minutes"].agg(
        [("max_bin", "max"), ("mean_bin", "mean")]).reset_index()

    # Calculate the load factor for each household-activity combination
    offpeak_hh["lf"] = offpeak_hh["mean_bin"] / offpeak_hh["max_bin"].replace(0, np.nan)

    # Aggregate to activity level
    offpeak_lf = offpeak_hh.groupby("activity_description")["lf"].agg(
        [("offpeak_lf_mean", "mean"), ("offpeak_lf_std", "std")]).reset_index()
    
    # Merge the results from all time periods into one table
    result = all_day_lf.merge(daytime_lf, on="activity_description", how="outer")
    result = result.merge(evening_lf, on="activity_description", how="outer")
    result = result.merge(offpeak_lf, on="activity_description", how="outer")
    
    # Return the combined results rounded to 3 decimal places
    return result.round(3)

In [ ]:
# Create empty lists to store typology-level results
results_lf = []
results_activity = []

# Group households by typology order and typology name
for (order, name), group in df1.groupby(["typo_order", "typo_name"]):
    # Extract the unique household IDs for the current typology
    mainids = group["mainid"].unique()

    # Count the number of households in the current typology
    n_hh = len(group)

    # Filter the expanded electricity/activity dataset to include only households
    # belonging to the current typology
    df_typo = df[df["mainid"].isin(mainids)]

    # Calculate the Load factors =======
    # Calculate activity load factors for the households in the current typology
    lf_df = calculate_activity_lf(
        df,
        mainids,
        day_engagement,
        evening_engagement,
        offpeak_engagement
    )

    # Add typology metadata to the load factor results
    lf_df.insert(0, "Order", order)
    lf_df.insert(1, "Typology", name)
    lf_df.insert(2, "N_households", n_hh)

    # Append the typology-specific load factor results to the list
    results_lf.append(lf_df)

    # Calculate Activity participation =======
    # Count the number of unique active households for each time bin and activity
    df_grouped = df_typo.groupby(
        ["time_bin", "starttime_of_day_min", "activity_description"]
    )["mainid"].nunique().reset_index()

    # Rename the grouped count column for clarity
    df_grouped.columns = [
        "time_bin",
        "starttime_of_day_min",
        "activity_description",
        "n_active"
    ]

    # Convert the number of active households into a participation percentage
    df_grouped["percentage_hh"] = (df_grouped["n_active"] / n_hh) * 100

    # Add typology metadata to the activity participation results
    df_grouped["Order"] = order
    df_grouped["Typology"] = name
    df_grouped["N_households"] = n_hh

    # Append the typology-specific activity participation results to the list
    results_activity.append(df_grouped)

In [ ]:
# Combine all load factor results into a single DataFrame
results_df = pd.concat(results_lf, ignore_index=True)

# Combine all activity participation results into a single DataFrame
df_typo_activity = pd.concat(results_activity, ignore_index=True)

In [ ]:
results_df, df_typo_activity

In [ ]:
# Pivot the table
results_wide = results_df.pivot_table(
    index=['Order', 'Typology', 'N_households'],
    columns='activity_description',
    values=['day_lf_mean', 'day_lf_std', 'daytime_lf_mean', 'daytime_lf_std',
            'evening_lf_mean', 'evening_lf_std', 'offpeak_lf_mean', 'offpeak_lf_std']
)

In [ ]:
# Flatten the column names
results_wide.columns = [f'{metric}_{activity}' for metric, activity in results_wide.columns]
results_wide = results_wide.reset_index()

In [ ]:
# Save the data
results_wide.to_csv('forced_typology_activities_wide.csv', index=False)

In [ ]:
# Extract the unique typology names from the results table
typology_list = results_df["Typology"].unique()

# Loop through each typology to create a separate bar chart
for typology in typology_list:
    # Filter the dataset for the current typology
    # and use activity description as the row index for plotting
    df_plot = results_df[results_df["Typology"] == typology].set_index("activity_description")

    # Get the number of households for the current typology
    # (assumed to be the same for all rows in this typology)
    n_hh = df_plot["N_households"].iloc[0]
    
    # Select the mean load factor columns for the different time periods
    df_means = df_plot[["day_lf_mean", "daytime_lf_mean", "evening_lf_mean", "offpeak_lf_mean"]]
    
    # Create a grouped bar chart of load factor means by activity
    ax = df_means.plot(kind="bar", figsize=(14, 4), width=0.7)

    # Add a title showing the typology name and household count
    ax.set_title(f"{typology}  (N = {n_hh} households)", fontsize=11)

    # Label the x-axis and y-axis
    ax.set_xlabel("Activity")
    ax.set_ylabel("Load Factor Mean")

    # Keep activity labels horizontal for readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")

    # Add readable legend labels for each time period
    ax.legend(["Whole Day", "Daytime", "Evening time", "Offpeak"],
              loc="best", bbox_to_anchor=(1, 0.5), frameon=True)

    plt.tight_layout()    # Adjust layout so labels and legend do not overlap

    # Save the figure using the typology name in the file name
    plt.savefig(f'lf_{typology.replace(" ", "_")}.png',
                dpi=150, bbox_inches="tight")

    # Display the chart
    plt.show()

==== High income

High income AND fully employed AND has young children:
The most rigid activity through out the whole day is laundry, but during the day engagement period food preparation is a rigid activity, while house cleaning becomes rigid in the evening. In the offpeak period, house cleaning becomes the least flexible.

High income AND fully employed AND No young Children
Across the whole day, dishwashing is the most rigid activity. In the day time engagement window, laundry is the most rigid, while dishwashing was rigid in the evening peak engagement period. Washing & Dressing is the most rigid in the off peak period.

High income AND partly employed AND Has young children
These households are rigid with home heating across the day and evening enagement period, but in the daytime  the rigid axtivities are dishwashing, while in the offpeak period, dishwashing, food preparation, house cleaning and laundry, are all perfectly rigid.

High income AND partly employed AND No young children
Home heating is the most rigid across the day, daytime and evening engagement periods, while diswashing is rigid in the offpeak engagement period.

High income AND fully retired AND No young children
Laundry is the most rigid activity across the day. In the day time engagement window, laundry, washing & dressing, and watching TV are perfectly rigid. Dishwashig and washing & dressing are perfectly rigid for the evening engagement and offpeak periods respectively.

High income AND partly retired AND has young children
House cleaning are rigid across the whole day, and in the morning, washing & dressing. Dishwashing and house cleaning are perfectly rigid in the evening engagement periods, while washing and dressing is perfectly rigid in the offpeak.

High income AND partly retired AND No young children
Across the day, laundry is the most rigid, while diswashing and house cleaning are more rigid in the day time engagement and evening enagement periods, respectively. The offpeak period have perfect rigidity for food preparation, washing & dressing, and watching TV.

High income AND partly unemployed AND has young children
There is only one household in this group. Their whole day rigidity is informed by food preparation. This activity and washing & dressing are perfectly rigid in the day time and evening engagement periods.

High income AND partly unemployed AND No young children
House cleaning is more rigid across the day, and eveining engagement period, but perfectly rigid in the day time period alongside washing & dressing. Washing & dressing is rigid during the offpeak period.

==== Medium Income

Medium income AND fully employed AND Has young children
Diswashing and laundry are more rigid across the whole day. Dishwashing is perfectly rigid in the day time and evening enagement periods, while watching TV maintains a perfect rigidity in the offpeak time.

Medium income AND fully employed AND No young children
Dishwashing and laundry are most rigid across the whole day, while dishwashing is perfectly rigid in the day time period. House cleaning and laundry have perfect rigidity in the evening engagement period, while food preparation and watching TV are perfectly rigid in the offpeak periods.

Medium income AND partly employed AND Has young children
Laundry is more rigid across the whole day, while dishwashing is perfectly rigid during the day time. Laundry and washng & dressing are more rigid in the evening engagement period. Dishwashing, house cleaning and laundry maintain perfect rigidity for the offpeak period.

Medium income AND partly employed AND No young children
Dishwashing is more rigid across the day, and home heating is perfectly rigid in the day time. Dishwashing, laundry, and washing & dressing are more rigid in the evening engagement period. These same activities are perfectly rigid in offpeak period.

Medium income AND fully retired AND Has young children
This group has only one household, and engaged on only two activities. Watching TV is more rigid across the day, and particularly in the evening engagement periods. Washing & dressing is rigid in the day time engagement period.

Medium income AND fully retired AND No young children
Heating house is perfectly rigid across the day, but not at specific periods. In the day time, heating house is perfectly rigid, while laundry is perfectly rigid in the evening engagement period. Washing & dressing and laundry are perfectly rigid in the offpeak period.

Medium income AND partly retired AND Has young children
Two households are present in this group. Dishwashing, laundry, and washing & dressing are all perfectly rigid across the whole day. Food preparation and laundry are perfectly rigid in the day time period, while dishwashing is perfectly rigid in the evening engagement period. Washing & dressing and watching TV are perfectly rigid in the offpeak period.

Medium income AND partly retired AND No young children
Dishwashing and house cleaning are more rigid across the whole day. Day time periods were rigid for dishwashing, and laundry was perfetly rigid for evening engagement periods. Dsihwashing is perfectlt rigid during the offpeak periods.

Medium income AND partly unemployed AND Has young children
House cleaning and dishwashing are more rigid across the day, while house cleaning, food preparation, and washing & dressing are more rigid in the day time period. Dishwashing and washing & dressing are perfectly rigid during the evening engagement period, and food preparation maintains perfect rigidity in the offpeak time window.

Medium income AND partly unemployed AND No young children
Laundry has a perfect rigidity across the whole day and day time period. Dishhwashing, house cleaning, laundry and washing & dressing are perfectly rigid in the evening engagement period, washing & dressing and watching TV are more rigid in the evening period.

==== Low income

Low income AND fully employed AND Has young children
House cleaning and dishwashing are more rigid across the day. ALthough dishwashing is perfectly rigid in the day time, and more rigid in the evening engagement period, wahtching TV maintains a perfect rigidity in the offpeak period.

Low income AND fully employed AND No young children
Food preparation and laundry are more rigid in across the day. Laundry maintains a perfect rigidity in the day time period, and food preparation in the evening engagement period. During the offpeak, food preparation maintains a perfect rigidity.

Low income AND partly employed AND Has young children
Dishwashing is more rigid in across the day, day time and evening engagement periods. It maintains a perfect rigidity in the offpeak alongside laundry, as well as a nearly perfect rigidity for food preparation for food preparation during the same period.

Low income AND partly employed AND No young children
Dishwashing and laundry are more rigid across the whole day. Dishwashing, food preparation, laundry and washing & dressing have high rigidity in the day time. Dishwashing, laundry, washing & dressing are more rigid in the evening engagement period. While food preparation and washing & dressing have high rigidity in the offpeak, dishwashing is perfectly rigid in the same period.

Low income AND fully retired AND No young children
Dishwashing, laundry, and washing & dressing are more rigid acrpss the day. Laundry is perfectly rigid across the day time, evening, and offpeak periods. Washing & dressing is also perfectly rigid in the offpeak period.

Low income AND partly retired AND Has young children
This group is made up of two households. Dishwashing and laundry are perfectly rigid across the whole day. While laundry is perfectly rigid in the day time, dishwashing maintains the same in the evening engegement period. There is no offpeak engagement for the group.

Low income AND partly retired AND No young children
Home heating and dishwashing are more rigid across the wholed day and the day time period. In the evening engagement period, dishwashing and washing & dressing are more rigid, while dishwashing and house cleaning are perfectly rigid in the offpeak.

Low income AND fully unemployed AND Has young children
Only one household is present here and they reported only watching TV, which is perfectly rigid throughout the day and day time period.

Low income AND fully unemployed AND No young children
House cleaning and laundry are perfectly rigid across the whole day. The daytime period showed perfect rigidity for dishwashing, food preparation, house cleaning, and laundry. While washing & dressing is perfectly rigid in the evening period, laundry and watching TV maintain same during offpeak periods.

Low income AND partly unemployed AND Has young children
Dishwashing has more rigidity across the day, with a perfect rigidity in the day time period. House cleaning is perfectly rigid in the evening engagement period, while it maintains a perfect rigidity in the offpeak alongside food preparation.

Low income AND partly unemployed AND No young children
Dishwashing and laundry are more rigid across the day. Dishwashing, food preparation, laundry, washing & dressing are more rigid in the day time, while dishwashing is rigid in the evening. Washing and dressing is more rigid in the offpeak period.

In [ ]:
activity_colors = {
    'Dishwashing':        "#1f77b4",  # blue
    'Food Preparation':   "#d62728",  # red
    'Heating House':      "#ffdd57",  # yellow
    'House Cleaning':     "#2ca02c",  # green
    'Laundry':            "#ff7f0e",  # orange
    'Washing & Dressing': "#9467bd",  # purple
    'Watching TV':        "#17becf"   # teal
}

typologies_list = df_typo_activity['Typology'].unique()

In [ ]:
# Loop through each typology to create a separate stacked area chart
for typology in typologies_list:
    # Filter the dataset to include only records for the current typology
    df_plot = df_typo_activity[df_typo_activity["Typology"] == typology].copy()

    # Sort the data by start time so the x-axis follows chronological order
    df_plot = df_plot.sort_values("starttime_of_day_min")

    # Get the number of households in the current typology
    n_hh = df_plot["N_households"].iloc[0]

    # Reshape the data so each row represents a time bin
    # and each column represents an activity
    df_pivot = df_plot.pivot_table(
        index="time_bin",
        columns="activity_description",
        values="percentage_hh",
        aggfunc="mean"
    )
    
    # Convert each row to percentages so that activity shares
    # sum to 100% within each time bin
    df_pivot = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

    # Replace missing values with 0 after normalisation
    df_pivot = df_pivot.fillna(0)

    # Skip typologies that do not have enough time points to plot meaningfully
    if len(df_pivot) < 2:
        print(f"Skipping {typology} — insufficient data")
        continue

    # Retrieve predefined colours for each activity
    # in the same order as the pivot table columns
    palette = [activity_colors[activity] for activity in df_pivot.columns]

    # Create the figure and axis for the stacked area chart
    fig, ax = plt.subplots(figsize=(14, 6))

    # Plot the activity distribution over time as a stacked area chart
    ax.stackplot(
        range(len(df_pivot)),
        df_pivot.T.values,
        labels=df_pivot.columns,
        colors=palette,
        alpha=0.85
    )

    # Set the x-axis range to cover all time bins
    ax.set_xlim(0, len(df_pivot) - 1)

    # Show one tick per time bin
    ax.set_xticks(range(len(df_pivot)))

    # Label the x-axis ticks using the time bin values
    ax.set_xticklabels(df_pivot.index, rotation=45, ha="right", fontsize=9)

    # Set the y-axis range from 0 to 100 percent
    ax.set_ylim(0, 100)

    # Define percentage tick marks for the y-axis
    yticks = [0, 20, 40, 60, 80, 100]
    ax.set_yticks(yticks)
    ax.set_yticklabels([f"{y}%" for y in yticks])

    # Add a descriptive title for the current typology
    # including the number of households
    ax.set_title(
        f"{typology}  (N = {n_hh} households)",
        fontsize=13,
        fontweight="bold"
    )

    # Label the x-axis and y-axis
    ax.set_xlabel("Time of Day", fontsize=12)
    ax.set_ylabel("Share of Activity (%)", fontsize=12)

    # Add a legend below the chart
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.3),
        ncol=3,
        frameon=True,
        title="Activity"
    )

    # Increase the bottom margin to make space for the legend
    plt.subplots_adjust(bottom=0.35)

    # Adjust layout to prevent overlap between labels and legend
    plt.tight_layout()

    # Save the figure using a cleaned file name
    plt.savefig(
        f"Activity_Distribution_{typology}.png".replace(" ", "_").replace("/", "_"),
        dpi=300,
        bbox_inches="tight"
    )

    # Display the chart
    plt.show()

    # Close the figure to free memory before the next iteration
    plt.close()